In [19]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.history import HISTORY_FEATURES
from src.modeling import NUMERIC_FEATURES, make_features
from src.neural_data import build_neural_preprocessor

In [20]:
history_features = pd.read_parquet(
    project_root / "data/processed/lab03_history_features.parquet"
)

datasets = {}

for split_name in ["train", "validation"]:
    base = pd.read_parquet(
        project_root / f"data/processed/lab02_{split_name}.parquet"
    )

    enriched = base.merge(
        history_features,
        on="source_row",
        how="left",
        sort=False,
        validate="one_to_one",
    )

    assert enriched["source_row"].equals(base["source_row"])
    assert enriched[HISTORY_FEATURES].notna().all().all()

    datasets[split_name] = enriched

In [21]:
extra_features = tuple(HISTORY_FEATURES)

numeric_types = {
    column: np.float32
    for column in NUMERIC_FEATURES + HISTORY_FEATURES
}

train_features = make_features(
    datasets["train"],
    extra_numeric_features=extra_features,
).astype(numeric_types)

validation_features = make_features(
    datasets["validation"],
    extra_numeric_features=extra_features,
).astype(numeric_types)

preprocessor = build_neural_preprocessor()

X_train = preprocessor.fit_transform(train_features)
X_validation = preprocessor.transform(validation_features)

y_train = datasets["train"]["is_laundering"].to_numpy(
    dtype=np.float32
)
y_validation = datasets["validation"]["is_laundering"].to_numpy(
    dtype=np.float32
)

In [22]:
assert X_train.shape == (1_379_551, 44)
assert X_validation.shape == (965_524, 44)

assert y_train.sum() == 1_800
assert y_validation.sum() == 1_036

for name, matrix in [
    ("train", X_train),
    ("validation", X_validation),
]:
    assert matrix.dtype == np.float32
    assert np.isfinite(matrix).all()

    print(
        name,
        matrix.shape,
        matrix.dtype,
        f"{matrix.nbytes / 1024**2:.1f} MiB",
    )

train (1379551, 44) float32 231.6 MiB
validation (965524, 44) float32 162.1 MiB


In [23]:
cache_dir = project_root / "data/processed/lab05"
cache_dir.mkdir(parents=True, exist_ok=True)

for name, array in {
    "X_train": X_train,
    "X_validation": X_validation,
    "y_train": y_train,
    "y_validation": y_validation,
}.items():
    np.save(cache_dir / f"{name}.npy", array)

for split_name, frame in datasets.items():
    np.save(
        cache_dir / f"{split_name}_source_rows.npy",
        frame["source_row"].to_numpy(),
    )

joblib.dump(
    preprocessor,
    cache_dir / "preprocessor.joblib",
)

['/Users/sara/Projects/data-science-lab/data/processed/lab05/preprocessor.joblib']

### Neural input preparation

We log-scaled amounts and counts, standardized numerical features,
and one-hot encoded categories. Preprocessing was fitted on training only.

The inputs contain 44 float32 columns. Training uses about 231.6 MiB;
validation uses about 162.1 MiB. These figures exclude other process
memory and temporary arrays.

We saved the arrays, source-row alignment, and fitted preprocessor.

In [24]:
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.neural import NeuralBaseline, predict_scores

torch.set_num_threads(2)

cache_dir = project_root / "data/processed/lab05"

X_train = torch.from_numpy(np.load(cache_dir / "X_train.npy"))
y_train = torch.from_numpy(np.load(cache_dir / "y_train.npy"))

X_validation = torch.from_numpy(
    np.load(cache_dir / "X_validation.npy")
)
y_validation = np.load(cache_dir / "y_validation.npy")

print("Training:", X_train.shape, X_train.dtype, X_train.device)
print("Validation:", X_validation.shape)
print(f"Training laundering rate: {y_train.mean().item():.3%}")

Training: torch.Size([1379551, 44]) torch.float32 cpu
Validation: torch.Size([965524, 44])
Training laundering rate: 0.130%


In [25]:
torch.manual_seed(42)

positive_rate = y_train.mean().item()

model = NeuralBaseline(
    n_features=X_train.shape[1],
    positive_rate=positive_rate,
)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=4096,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

epochs = 5
training_history = []

for epoch in range(1, epochs + 1):
    start = perf_counter()
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)

    validation_scores = predict_scores(model, X_validation)
    validation_ap = average_precision_score(
        y_validation,
        validation_scores,
    )

    result = {
        "epoch": epoch,
        "train_loss": total_loss / len(y_train),
        "validation_ap": validation_ap,
        "seconds": perf_counter() - start,
    }
    training_history.append(result)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"loss={result['train_loss']:.5f} | "
        f"validation AP={validation_ap:.4f} | "
        f"{result['seconds']:.1f}s",
        flush=True,
    )

history = pd.DataFrame(training_history)
history

Epoch 1/5 | loss=0.00814 | validation AP=0.0346 | 5.8s
Epoch 2/5 | loss=0.00696 | validation AP=0.0504 | 5.5s
Epoch 3/5 | loss=0.00678 | validation AP=0.0565 | 5.5s
Epoch 4/5 | loss=0.00667 | validation AP=0.0728 | 5.7s
Epoch 5/5 | loss=0.00660 | validation AP=0.0821 | 5.4s


,epoch,train_loss,validation_ap,seconds
0,1,0.008140,0.034639,5.833139
1,2,0.006958,0.050445,5.549923
2,3,0.006781,0.056540,5.480189
3,4,0.006673,0.072814,5.655866
4,5,0.006602,0.082083,5.390349


In [26]:
reports_dir = project_root / "reports"
models_dir = project_root / "models"

reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

history.to_csv(
    reports_dir / "lab_05_training.csv",
    index=False,
)

np.save(
    cache_dir / "validation_scores.npy",
    validation_scores,
)

torch.save(
    {
        "model_state": model.state_dict(),
        "n_features": X_train.shape[1],
        "positive_rate": positive_rate,
        "epochs": epochs,
    },
    models_dir / "lab05_neural_baseline.pt",
)

In [27]:
from src.modeling import evaluate_scores

validation = pd.read_parquet(
    project_root / "data/processed/lab02_validation.parquet",
    columns=["source_row", "timestamp", "is_laundering"],
)

cache_dir = project_root / "data/processed/lab05"

scores = np.load(cache_dir / "validation_scores.npy")
saved_rows = np.load(cache_dir / "validation_source_rows.npy")
saved_labels = np.load(cache_dir / "y_validation.npy")

# Matching lengths alone does not guarantee matching row order.
np.testing.assert_array_equal(
    validation["source_row"].to_numpy(),
    saved_rows,
)
np.testing.assert_array_equal(
    validation["is_laundering"].to_numpy(),
    saved_labels,
)

neural_metrics = evaluate_scores(
    validation,
    scores,
    daily_budget=50,
)

reference = pd.read_csv(
    project_root / "reports/lab_03_history_metrics.csv",
    index_col=0,
)

comparison = reference.loc[
    ["transaction_only", "with_history"],
    list(neural_metrics),
].rename(index={
    "transaction_only": "boosting_transaction_only",
    "with_history": "boosting_with_history",
})

comparison.loc["neural_with_history"] = pd.Series(neural_metrics)
comparison.index.name = "model"

comparison.round(4)

,average_precision,alerts,laundering_caught,precision_at_budget,recall_at_budget
model,,,,,
boosting_transaction_only,0.0669,100.0,27.0,0.27,0.0261
boosting_with_history,0.1338,100.0,52.0,0.52,0.0502
neural_with_history,0.0821,100.0,35.0,0.35,0.0338
